# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

## Load Data

In [ ]:
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)
print(f"{len(df)} rows, {df.shape[1]} columns: {list(df.columns)}")
df.head()

## (Optional) Filter by Product Type
Set `PRODUCT_TYPE` below to focus on a specific product, or leave it as `None` to use all products. Run the cell to see all available types.

In [ ]:
# ---- Choose a product type (or None for all) ----
PRODUCT_TYPE = None  # e.g. 'T-shirt', 'Dress', 'Sweater', 'Trousers'

# Show all available product types
print("Available product types:")
print(sorted(df['name'].unique()))

if PRODUCT_TYPE:
    assert PRODUCT_TYPE in df['name'].values, f"'{PRODUCT_TYPE}' not found. See list above."
    df = df[df['name'] == PRODUCT_TYPE].copy()
    print(f"\nFiltered to '{PRODUCT_TYPE}': {len(df)} rows, {df['id'].nunique()} products")
else:
    print(f"\nUsing all product types: {len(df)} rows, {df['id'].nunique()} products")

## Train/Test Split by Product
We hold out 20% of product IDs as our test set. The model never sees these products during training.

In [ ]:
np.random.seed(42)
product_ids = df['id'].unique()
np.random.shuffle(product_ids)
split = int(0.8 * len(product_ids))
train_ids, test_ids = product_ids[:split], product_ids[split:]

print(f"Train: {len(train_ids)} products, Test: {len(test_ids)} products")

---
# Block 1: Basic Linear Regression
Start simple — use only **price** and **month** to predict sales.

In [ ]:
# Months are already one-hot encoded in the CSV
month_cols = ['April', 'August', 'December', 'February', 'January',
              'July', 'June', 'March', 'May', 'November', 'October', 'September']

basic_features = ['price'] + month_cols
print(f"Features ({len(basic_features)}): {basic_features}")

In [ ]:
df_train = df[df['id'].isin(train_ids)]
df_test  = df[df['id'].isin(test_ids)]

X_train = df_train[basic_features].values
X_test  = df_test[basic_features].values
y_train = df_train['sales'].values
y_test  = df_test['sales'].values

scaler1 = StandardScaler()
X_train_s = scaler1.fit_transform(X_train)
X_test_s  = scaler1.transform(X_test)

ols_basic = LinearRegression().fit(X_train_s, y_train)
mae_basic = mean_absolute_error(y_test, ols_basic.predict(X_test_s))
print(f"OLS (price + month) — Test MAE: {mae_basic:.1f}")

---
# Block 2: Feature Engineering → Overfitting Risk
Add lag features, rolling statistics, polynomial and interaction terms. With many features and plain OLS, we risk **overfitting** — the model memorizes training noise instead of learning real patterns.

In [ ]:
def engineer_features(data):
    """Add engineered features within each product's time series."""
    out = data.copy()
    
    # Lag features (per product)
    for i in range(1, 7):
        out[f'lag_{i}'] = out.groupby('id')['sales'].shift(i)
    
    # Rolling statistics (per product)
    for w in [3, 5]:
        out[f'ma_{w}']  = out.groupby('id')['sales'].transform(lambda x: x.rolling(w).mean().shift(1))
        out[f'std_{w}'] = out.groupby('id')['sales'].transform(lambda x: x.rolling(w).std().shift(1))
    
    # Polynomial terms
    out['price_sq'] = out['price'] ** 2
    for i in range(1, 4):
        out[f'lag_{i}_sq'] = out[f'lag_{i}'] ** 2
    
    # Interaction terms
    for i in range(1, 4):
        out[f'price_x_lag_{i}'] = out['price'] * out[f'lag_{i}']
    out['lag1_x_lag2'] = out['lag_1'] * out['lag_2']
    
    # Price change
    out['price_change'] = out.groupby('id')['price'].diff()
    
    out.fillna(0, inplace=True)
    out.replace([np.inf, -np.inf], 0, inplace=True)
    return out

df_eng = engineer_features(df)

# Collect all feature names
exclude = {'id', 'sales', 'name'}
all_features = [c for c in df_eng.columns if c not in exclude
                and df_eng[c].dtype in ['float64', 'int64', 'int32', 'uint8']
                and df_eng[c].std() > 0]
print(f"{len(all_features)} features")

In [ ]:
df_train_eng = df_eng[df_eng['id'].isin(train_ids)]
df_test_eng  = df_eng[df_eng['id'].isin(test_ids)]

X_train2 = df_train_eng[all_features].values
X_test2  = df_test_eng[all_features].values
y_train2 = df_train_eng['sales'].values
y_test2  = df_test_eng['sales'].values

scaler2 = StandardScaler()
X_train2_s = scaler2.fit_transform(X_train2)
X_test2_s  = scaler2.transform(X_test2)

ols_eng = LinearRegression().fit(X_train2_s, y_train2)
mae_eng_train = mean_absolute_error(y_train2, ols_eng.predict(X_train2_s))
mae_eng_test  = mean_absolute_error(y_test2, ols_eng.predict(X_test2_s))

print(f"OLS ({len(all_features)} features)")
print(f"  Train MAE: {mae_eng_train:.1f}")
print(f"  Test  MAE: {mae_eng_test:.1f}")
print(f"\n  Train/test gap: {mae_eng_test - mae_eng_train:.1f}  (large gap = overfitting)")

---
# Block 3: Regularization to the Rescue
Use **Lasso** (L1) and **Ridge** (L2) to tame the large feature set. Regularization penalizes large coefficients, reducing overfitting and often improving test performance.

### Lasso (L1): Shrinks + eliminates coefficients

In [ ]:
alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
lasso_results = []

for a in alphas:
    m = Lasso(alpha=a, max_iter=10000).fit(X_train2_s, y_train2)
    lasso_results.append({
        'alpha': a,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2_s)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2_s)),
        'nonzero':   int(np.sum(m.coef_ != 0))
    })

lasso_df = pd.DataFrame(lasso_results)
print(lasso_df.to_string(index=False))

### Ridge (L2): Shrinks coefficients but keeps them all

In [ ]:
ridge_results = []

for a in alphas:
    m = Ridge(alpha=a).fit(X_train2_s, y_train2)
    ridge_results.append({
        'alpha': a,
        'Train MAE': mean_absolute_error(y_train2, m.predict(X_train2_s)),
        'Test MAE':  mean_absolute_error(y_test2, m.predict(X_test2_s)),
    })

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.to_string(index=False))

### Summary

In [ ]:
# Best alpha for each method
best_lasso_a = lasso_df.loc[lasso_df['Test MAE'].idxmin(), 'alpha']
best_ridge_a = ridge_df.loc[ridge_df['Test MAE'].idxmin(), 'alpha']

lasso_best = Lasso(alpha=best_lasso_a, max_iter=10000).fit(X_train2_s, y_train2)
ridge_best = Ridge(alpha=best_ridge_a).fit(X_train2_s, y_train2)

results = pd.DataFrame([
    {'Model': 'OLS (price + month)',                  'Test MAE': mae_basic},
    {'Model': f'OLS ({len(all_features)} features)',  'Test MAE': mae_eng_test},
    {'Model': f'Lasso (\u03B1={best_lasso_a})',       'Test MAE': mean_absolute_error(y_test2, lasso_best.predict(X_test2_s))},
    {'Model': f'Ridge (\u03B1={best_ridge_a})',       'Test MAE': mean_absolute_error(y_test2, ridge_best.predict(X_test2_s))},
])
print(results.to_string(index=False))

In [ ]:
colors = ['#888', '#1f77b4', '#ff7f0e', '#2ca02c']
bars = plt.bar(results['Model'], results['Test MAE'], color=colors, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, results['Test MAE']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.0f}',
             ha='center', va='bottom', fontsize=11)
plt.ylabel('Test MAE')
plt.title('Test MAE Comparison')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()